# Workshop: Declarative Automation Bundles — Deploy RetailHub

**Learning objective:** deploy the RetailHub capstone (Lakeflow Spark Declarative Pipeline + 3-task Lakeflow Job) to a `dev` target as a **Declarative Automation Bundle** (formerly *Databricks Asset Bundles / DABs*), and verify the deployment programmatically.

**Expected duration:** ~30 min (guided)

**Prerequisites:**
- lab_07 completed (the pipeline sources in `materials/lakeflow/lakeflow_demo` are familiar)
- The repo available as a **Git folder** in your workspace
- CLI access for tasks 3 & 5 — **web terminal** or local Databricks CLI. No CLI? Use the documented **trainer-driven fallback** in Task 3.

| Task | Topic |
|------|-------|
| 1 | Inspect `databricks.yml` — bundle anatomy |
| 2 | Compose the deploy command with your catalog variable |
| *opt.* | Git folder: branch, commit & push, pull request (trainer-dependent) |
| 3 | `databricks bundle validate` / `deploy` (CLI or trainer-driven) |
| 4 | Verify the deployment from this notebook (Databricks SDK) |
| 5 | Run the job and confirm a terminal SUCCESS state |
| 6 | Reflection — `dev` vs `prod` targets |

## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
import os

# The bundle lives in materials/cicd/ — three levels up from this lab notebook
BUNDLE_DIR = os.path.abspath(os.path.join(os.getcwd(), "../../../materials/cicd"))
print(f"Bundle root: {BUNDLE_DIR}\n")

try:
    databricks_yml = open(f"{BUNDLE_DIR}/databricks.yml").read()
    print(databricks_yml)
except Exception as e:
    databricks_yml = None
    print("Could not read the file from here — open materials/cicd/databricks.yml in the workspace file browser instead.")
    print(e)

## Task 1: Inspect `databricks.yml` — bundle anatomy

Read the bundle root file printed above (or open `materials/cicd/databricks.yml`) and fill in the
`bundle_structure` dict describing what you found.

**What you need to do:** fill in the bundle name, the default target, that target's mode, and the
list of declared variable names.

**Guidance — Task 01**

A bundle root file (`databricks.yml`) has four top-level areas you should be able to read cold on the exam:

- **`bundle:`** — the bundle `name`, namespacing everything the bundle deploys.
- **`include:`** — additional YAML files merged into the config (here: `resources/*.yml`, one file per resource).
- **`variables:`** — declared inputs with optional `default`s; referenced as `${var.<name>}` and overridden per deploy with `--var="name=value"`.
- **`targets:`** — named environments. Exactly one can carry `default: true`; `mode: development` gives dev-loop behavior (name prefixes, paused triggers), `mode: production` is strict.

Also note `sync.paths` — this bundle pulls in source folders that live *outside* the bundle root
(`../lakeflow/lakeflow_demo`, `../orchestration`), so `bundle deploy` uploads them too.

In [0]:
# TODO: Fill in the dict by reading databricks.yml (printed in the Setup cell)
bundle_structure = {
    "bundle_name":         ...,   # bundle.name
    "default_target":      ...,   # which target has default: true?
    "default_target_mode": ...,   # that target's mode
    "variable_names":      [...], # names of ALL declared variables
}
print(bundle_structure)

In [0]:
# -- Validation --
assert bundle_structure["bundle_name"] == "retailhub", "Check bundle.name in databricks.yml"
assert bundle_structure["default_target"] == "dev", "Which target carries default: true?"
assert bundle_structure["default_target_mode"] == "development", "Check the mode of the dev target"
assert sorted(bundle_structure["variable_names"]) == ["catalog", "schema_prefix"], \
    f"Expected the two declared variables, got: {bundle_structure['variable_names']}"
print("Task 1 OK: bundle anatomy understood")

## Task 2: Compose the deploy command

The bundle's `catalog` variable defaults to `retailhub_trainer` — deploying without overriding it
would target the trainer's catalog (and fail on permissions). Compose the exact deploy command
**for your own catalog** as a Python string.

**What you need to do:** build `deploy_command` — a `databricks bundle deploy` invocation targeting
the `dev` target and overriding the `catalog` variable with **your** `CATALOG` value.

**Guidance — Task 02**

**Command shape**

```
databricks bundle deploy -t <target> --var="<name>=<value>"
```

- `-t dev` selects the target (it is the default here, but being explicit is good CI hygiene).
- `--var="catalog=retailhub_jan_kowalski"` overrides a declared variable for this invocation.
  Multiple variables → repeat the flag.
- Your catalog name is already in the `CATALOG` variable exported by `00_setup` — use an f-string.

The same variable can also be set via environment (`BUNDLE_VAR_catalog`) or per-target
`variables:` overrides in `databricks.yml` — the `--var` flag wins over defaults.

In [0]:
# TODO: Compose the deploy command using YOUR catalog (the CATALOG variable)
# Shape: databricks bundle deploy -t dev --var="catalog=<your catalog>"
deploy_command = ...  # YOUR CODE HERE (f-string)

print(deploy_command)

In [0]:
# -- Validation --
assert isinstance(deploy_command, str), "deploy_command must be a string"
assert deploy_command.startswith("databricks bundle deploy"), "Start with: databricks bundle deploy"
assert "-t dev" in deploy_command, "Target the dev target explicitly: -t dev"
assert f"catalog={CATALOG}" in deploy_command, "Override the catalog variable with YOUR catalog (use CATALOG)"
print(f"Task 2 OK: {deploy_command}")

## Optional — Git folder workflow before deploying (objective 5.1)

> 👨‍🏫 **Trainer-dependent.** Needs a Git folder connected to a Git provider where you can push (Git credentials set in **Settings → Linked accounts**). If the provider is not connected, read the steps and continue with Task 3 — nothing below is checked by an assert.

1. **Workspace** → your Git folder → click the **branch name** → Git dialog → **Create Branch** `feature/<your_name>` from `main`.
2. Open `materials/cicd/resources/retailhub_job.yml` and make a harmless change, e.g. extend the `description` or add
   ```yaml
         tags:
           owner: <your_name>
   ```
   (at the same indentation as `max_concurrent_runs`).
3. Git dialog → check the diff → commit message `lab09: tag retailhub_job` → **Commit & Push**.
4. In the Git provider open a **pull request** `feature/<your_name>` → `main` (do not merge unless the trainer says so).
5. Stay on your feature branch: Task 3 deploys **what is checked out in the Git folder**, so your change ends up in the deployed job — a mini CI/CD loop.


## Task 3: Validate & deploy the bundle (CLI)

> 🖥️ **These commands run in a terminal, not in this notebook.** Pick ONE of the options below,
> then record what you observed in the answer cell.

### Option A — Web terminal (preferred)
1. Open the web terminal (attached compute → **Terminal**, or the terminal icon in the bottom panel).
2. `cd` into the bundle root inside your Git folder, then validate and deploy:

```bash
cd /Workspace/Users/<your login>/<git folder name>/materials/cicd

databricks bundle validate -t dev --var="catalog=<YOUR CATALOG>"
# read the summary: name, target, workspace host, user, path

databricks bundle deploy   -t dev --var="catalog=<YOUR CATALOG>"   # <- Task 2 command
```

### Option B — Local CLI
Clone the repo locally, authenticate (`databricks auth login --host <workspace-url> --profile TRAINING`),
run the same two commands from `materials/cicd/` with `-p TRAINING`.

### Option C — 👨‍🏫 Trainer-driven fallback (no CLI available)
The trainer runs `validate` + `deploy` on the shared screen. **Follow with this observation checklist:**

- [ ] `validate` prints the bundle **name** (`retailhub`) and **target** (`dev`)
- [ ] `validate` prints the **workspace path** the bundle will deploy to (`/Workspace/Users/<user>/.bundle/retailhub/dev`)
- [ ] `validate` ends with **no errors** ("Validation OK")
- [ ] `deploy` uploads files then reports **deployment complete**
- [ ] In **Jobs & Pipelines** a job **`[dev <user>] retailhub_job`** and a pipeline **`[dev <user>] retailhub_pipeline`** appeared
- [ ] The `[dev …]` prefix and paused schedule come from `mode: development`

Then answer Task 4/5 against the trainer's deployed job (you have view access).

In [0]:
# TODO: Record how it went (this is your lab log — be honest, the assert checks it)
task3_result = {
    "how_i_ran_it":       ...,  # "web_terminal" | "local_cli" | "trainer"
    "validate_succeeded": ...,  # did `bundle validate -t dev` finish without errors? True/False
}
print(task3_result)

In [0]:
# -- Validation --
assert task3_result["how_i_ran_it"] in ("web_terminal", "local_cli", "trainer"), \
    "Use one of: web_terminal, local_cli, trainer"
assert task3_result["validate_succeeded"] is True, \
    "bundle validate must pass before deploying — ask the trainer if it failed"
print("Task 3 OK: bundle validated and deployed via", task3_result["how_i_ran_it"])

## Task 4: Verify the deployment from this notebook

Prove the deploy worked **programmatically** — no UI clicking. Use the Databricks SDK
(`databricks.sdk`, pre-installed on Databricks compute) to list jobs and find the deployed
RetailHub job.

**What you need to do:** build `retailhub_jobs` — all jobs visible to you whose name contains
`retailhub_job`. Remember: `mode: development` deployed it as **`[dev <user>] retailhub_job`**.

**Guidance — Task 04**

**WorkspaceClient without arguments**
Inside a Databricks notebook, `WorkspaceClient()` authenticates automatically from the runtime context
— no host/token needed.

**Listing jobs**

```python
for j in w.jobs.list():
    print(j.job_id, j.settings.name)
```

`w.jobs.list()` returns an iterator of `BaseJob`; the display name is `j.settings.name`.
A list comprehension with a substring test on the name is all you need.

**Why the `[dev …]` prefix?** `mode: development` prefixes every resource name with
`[dev <your user name>]` so many people can deploy the same bundle into one workspace
without collisions. In `mode: production` the name stays exactly `retailhub_job`.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# TODO: list all jobs you can see whose name contains "retailhub_job"
retailhub_jobs = [
    # YOUR CODE HERE — iterate w.jobs.list(), filter on j.settings.name
]

for j in retailhub_jobs:
    print(f"{j.job_id}  {j.settings.name}")

In [0]:
# -- Validation --
assert len(retailhub_jobs) >= 1, "No deployed retailhub_job found — did Task 3 deploy succeed?"
_names = [j.settings.name for j in retailhub_jobs]
assert any(n.startswith("[dev") and "retailhub_job" in n for n in _names), \
    f"Expected a dev-mode name like '[dev <user>] retailhub_job', got: {_names}"

# Prefer your own copy if several participants deployed into this workspace
_me = w.current_user.me().user_name.split("@")[0].replace(".", "_").lower()
_mine = [j for j in retailhub_jobs if _me in j.settings.name.lower()] or retailhub_jobs
retailhub_job_id = _mine[0].job_id
print(f"Task 4 OK: found {_names} — using job_id={retailhub_job_id}")

## Task 5: Run the job and confirm it succeeded

Trigger a run of the deployed job, then confirm from this notebook that it reached a **terminal
SUCCESS state** via the SDK.

**Start the run — pick one:**

```bash
# CLI (web terminal or local) — streams task states until the job finishes:
databricks bundle run -t dev --var="catalog=<YOUR CATALOG>" retailhub_job
```

- …or **Jobs & Pipelines → `[dev …] retailhub_job` → Run now** (allowed — the point of the lab is the bundle, not avoiding the UI),
- …or 👨‍🏫 the **trainer** runs the shared deployment and you verify that run.

⏱️ The run takes a few minutes (validate → pipeline refresh → report). Run the check cell below,
and re-run it until the run leaves the `RUNNING` state.

**Guidance — Task 05**

**Fetching runs for one job**

```python
runs = list(w.jobs.list_runs(job_id=<id>, limit=5))
latest = runs[0]                      # newest first
```

**Run state anatomy** — two different fields, a classic exam trap:
- `run.state.life_cycle_state` — where the run is in its lifecycle: `PENDING → RUNNING → TERMINATED`
- `run.state.result_state` — **only set once terminated**: `SUCCESS`, `FAILED`, `CANCELED`, `TIMEDOUT`

So: a healthy finished run has `life_cycle_state = TERMINATED` **and** `result_state = SUCCESS`.
Both are enums — use `.value` to get the string, and guard for `None` while the run is still going.

In [0]:
# TODO: fetch the latest run of retailhub_job_id and extract its result state
runs = list(w.jobs.list_runs(job_id=retailhub_job_id, limit=5))

latest_run   = ...   # YOUR CODE HERE — the most recent run
result_state = ...   # YOUR CODE HERE — its result state as a string ("SUCCESS", ...)
                     # hint: latest_run.state.result_state, None until the run terminates

print(f"life_cycle_state = {latest_run.state.life_cycle_state.value if latest_run.state else '?'}")
print(f"result_state     = {result_state}")

In [0]:
# -- Validation --
assert len(runs) >= 1, "No runs found — trigger the job first (bundle run / Run now / trainer)"
assert result_state is not None, \
    "Run has not reached a terminal state yet — wait a bit and re-run the previous cell + this one"
assert str(result_state).upper() == "SUCCESS", \
    f"Run terminated with {result_state} — open the run in Jobs & Pipelines and check the failed task"
print(f"Task 5 OK: run {latest_run.run_id} finished with result_state=SUCCESS")

## Task 6: Reflection — `dev` vs `prod` targets

Look at the commented `prod` stub at the bottom of `databricks.yml` and at what `mode: development`
did to your deployment, then fill in the reflection dict.

**Guidance — Task 06**

Compare what you saw with what the `prod` stub declares:

| Aspect | `dev` (`mode: development`) | `prod` (`mode: production`) |
|---|---|---|
| Resource names | prefixed | exact names from YAML |
| Schedules & triggers | paused automatically | active |
| Deploy path | your user home (`/Users/<you>/.bundle/...`) | shared root path |
| Identity | you | typically a **service principal** (`run_as`) |
| Code | **the same YAML + sources** | the same — only target config differs |

The last row is the core CI/CD idea (and an exam favorite): promotion between environments changes
**target configuration**, never the code.

In [0]:
# TODO: fill in the reflection
reflection = {
    "dev_name_prefix":       ...,  # what does dev mode prepend to resource names? (string, e.g. "[dev <user>] ")
    "dev_pauses_triggers":   ...,  # are schedules/triggers paused in a development-mode deployment? True/False
    "prod_runs_as":          ...,  # recommended identity for prod deployments: "personal_user" | "service_principal"
    "promotion_changes":     ...,  # what changes when promoting dev -> prod: "the_code" | "only_target_config"
}
print(reflection)

In [0]:
# -- Validation --
assert "[dev" in str(reflection["dev_name_prefix"]), \
    "Look at your deployed job's name in Jobs & Pipelines"
assert reflection["dev_pauses_triggers"] is True, \
    "development mode pauses schedules/triggers so dev copies never fire on their own"
assert reflection["prod_runs_as"] == "service_principal", \
    "Production deployments should not depend on a person's identity"
assert reflection["promotion_changes"] == "only_target_config", \
    "Same YAML + sources deploy everywhere; only the target section differs"
print("Task 6 OK: dev vs prod understood")

## Summary

| Task | Topic | Key Point |
|------|-------|-----------|
| 1 | Bundle anatomy | `bundle` / `include` / `variables` / `targets` (+ `sync.paths` for outside sources) |
| 2 | Variables | `${var.catalog}` in YAML, overridden per deploy with `--var="catalog=..."` |
| 3 | CLI flow | `validate` → `deploy` → resources appear; `mode: development` = `[dev …]` prefix + paused triggers |
| 4 | SDK verification | `WorkspaceClient().jobs.list()` — verify deployments programmatically, not by clicking |
| 5 | Run states | `life_cycle_state` (PENDING/RUNNING/TERMINATED) vs `result_state` (SUCCESS/FAILED/…) |
| 6 | dev vs prod | Promotion changes target config (identity, paths, variables) — never the code |

**Stretch goals (if time permits):**
- Re-point the `publish_report` task at a RetailHub gold table (edit `resources/retailhub_job.yml`
  — note it needs a RetailHub-aware report notebook, see the comment in the YAML) and redeploy.
- Tear your deployment down: `databricks bundle destroy -t dev --var="catalog=<YOUR CATALOG>"`
  — then check Jobs & Pipelines to confirm both resources are gone.

> 🎯 **Exam (Implementing CI/CD):** Git folder branch/commit/push/PR flow, bundle file anatomy, `validate`/`deploy`/`run`/`destroy`,
> targets & modes, variable overrides, and why bundles beat manual UI configuration
> (versioned, reviewable, repeatable, environment-promotable).

← [09 — CI/CD & Automation](../demo/09_cicd_and_automation.ipynb) | **[README](../../../README.md)** | [Lab — Troubleshooting →](lab_troubleshooting.ipynb)